# Tome C1, Chapter 3: Counting (Python companion)

Exercises first, solutions at the end. A formula is a claim, and a computer is a merciless referee: here every count of the chapter is checked against brute force. We compute Möbius functions of arbitrary posets, encode and decode trees as Prüfer words and sample random trees, generate random plane trees with the cycle lemma, count graphs and trees up to symmetry with Pólya's method, settle Cayley's alkanes up to thirty carbons, and count subspaces of $\mathbb F_2^n$ and invertible binary matrices.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, heapq, time
from collections import Counter, defaultdict
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
random.seed(3)

## Exercises

**P3.1 (★) The sieve loop.** Count the steps of the loop `for d in 1..n: for k in multiples of d` and compare with $\sum_{k\le n}\tau(k)$ and with $n\ln n$. Plot the average number of divisors of the integers up to $n$ minus $\ln n$. Does it converge? (Dirichlet found the constant $2\gamma-1$; Tome 1, Chapter 4.)

**P3.2 (★★) Möbius functions by machine.** Write a function that computes $\mu(x,y)$ for any finite poset given by its order relation, with the recursion of Definition 3.2. Check it against Proposition 3.4 on chains, Boolean lattices and divisor lattices, against Philip Hall's chain formula (Proposition 3.5), and compute $\mu(\hat0,\hat1)$ for the partition lattices $\Pi_n$, $n\le6$ (Exercise 3.2).

**P3.3 (★★) Primitive words and Lyndon words.** Count primitive binary words by brute force and compare with $M(n)=\sum_{d\mid n}\mu(n/d)2^d$. Generate the Lyndon words with Duval's algorithm, check the count $M(n)/n$ (Exercise 3.3), and compare with the number of irreducible polynomials of degree $n$ over $\mathbb F_2$ found by brute force (Tome 2, Theorem 9.10).

**P3.4 (★★) Prüfer codes and random trees.** Implement encoding and decoding in $O(n\log n)$ with a heap. Check that decoding all $n^{n-2}$ words gives $n^{n-2}$ distinct trees and that encoding inverts decoding (Theorem 3.9) for $n\le7$. Then sample random trees with $n=10^5$ vertices: what fraction are leaves (Exercise 3.4)? What is the distribution of the degree of a vertex?

**P3.5 (★★) The cycle lemma at work.** Check the cycle lemma (Theorem 3.11) on random sequences. Then use it to sample a *uniformly random* plane tree with $m$ vertices in linear time: shuffle a multiset of numbers of children, rotate to the good shift, decode the Łukasiewicz word (Proposition 3.13). Test uniformity for $m=5$ (there are $C_4=14$ plane trees), and parse a prefix expression with the counter $h_j$.

**P3.6 (★★) Graphs up to isomorphism.** Compute the cycle index of $S_n$ acting on pairs by running through all permutations, for $n\le6$, and the numbers of graphs by number of edges. Compare with a brute-force classification by canonical forms for $n\le5$ (Exercise 3.8, Figure 3.4).

**P3.7 (★★★) Cayley's trees and alkanes.** (a) Compute the rooted trees $t_n$ with the recursion of Exercise 3.9, and check them for $n\le10$ by generating all rooted trees as canonical strings. (b) Compute the ternary trees $r_n$ (alcohols) and the alkanes $a_n$ with Theorem 3.16, and check $a_n$ for $n\le12$ by generating all trees of maximum degree $4$ up to isomorphism, using Jordan's centroid (Lemma 3.15) to make the canonical form. How many isomers do $\mathrm C_{20}\mathrm H_{42}$ and $\mathrm C_{30}\mathrm H_{62}$ have?

**P3.8 (★★) Vector spaces over $\mathbb F_2$.** Compute $\binom nk_q$ as polynomials, check the inversion formula (Theorem 3.19) and the $q$-binomial theorem (Theorem 3.20) symbolically, and count the subspaces of $\mathbb F_2^n$ by brute force for $n\le5$ (Theorem 3.18). Then estimate the probability that a random $n\times n$ binary matrix is invertible (Exercise 3.11), with a bit-packed Gaussian elimination.

**P3.9 (★) Ménages.** Count the seatings of Exercise 3.12 by brute force for $n\le8$ and compare with Touchard's formula.

## Solutions

### P3.1
**How to find it.** The inner loop runs $\lfloor n/d\rfloor$ times; the double count of Section 3.1 says the total is also $\sum_k\tau(k)$, so the loop doubles as a divisor-counting sieve.

In [ ]:
def sieve_tau(n):
    tau = [0] * (n + 1); steps = 0
    for d in range(1, n + 1):
        for k in range(d, n + 1, d):
            tau[k] += 1; steps += 1
    return tau, steps

for n in [10, 100, 1000, 10**5]:
    tau, steps = sieve_tau(n)
    print(f"n={n:>6}: steps {steps:>8} = sum tau {sum(tau):>8} = sum floor(n/d) {sum(n // d for d in range(1, n + 1)):>8};"
          f"  n ln n = {n * math.log(n):10.0f}")

N = 10**6
tau = np.zeros(N + 1, dtype=np.int64)
for d in range(1, N + 1):
    tau[d::d] += 1
ns = np.arange(1, N + 1)
avg = np.cumsum(tau[1:]) / ns
plt.figure(figsize=(7, 3))
plt.semilogx(ns[9:], avg[9:] - np.log(ns[9:]), lw=1)
plt.axhline(2 * np.euler_gamma - 1, color='C1', ls='--', label=r'$2\gamma-1$')
plt.xlabel('n'); plt.ylabel(r'average $\tau$ minus $\ln n$'); plt.legend(); plt.tight_layout(); plt.show()

The difference settles near $2\gamma-1\approx0.1544$, with an error of order $1/\sqrt n$ (Dirichlet's hyperbola method; making the error smaller is the *Dirichlet divisor problem*, still open).

### P3.2
**How to find it.** Store the poset as a list of elements and a function `leq`; compute $\mu(x,\cdot)$ from $x$ upwards in a linear extension, using $\mu(x,y)=-\sum_{x\le z<y}\mu(x,z)$. Partitions of a set can be stored as frozensets of frozensets; $\pi\le\sigma$ when each block of $\pi$ lies in a block of $\sigma$.

In [ ]:
def mobius(elements, leq, x):
    # mu(x, y) for all y >= x, by the recursion of Definition 3.2
    up = [y for y in elements if leq(x, y)]
    up.sort(key=lambda y: sum(leq(z, y) for z in up))   # sizes of down-sets: a linear extension
    mu = {}
    for y in up:
        mu[y] = 1 if y == x else -sum(mu[z] for z in up if z != y and leq(z, y) and z in mu)
    return mu

# chains
ch = list(range(6)); mu = mobius(ch, lambda a, b: a <= b, 0)
print('chain:', [mu[j] for j in ch])
# Boolean lattice B_4: subsets as bit masks
B = list(range(16)); mu = mobius(B, lambda a, b: a & b == a, 0)
assert all(mu[T] == (-1) ** bin(T).count('1') for T in B)
# divisors of 360: compare with sympy's classical Moebius function
D = sp.divisors(360); mu = mobius(D, lambda a, b: b % a == 0, 1)
assert all(mu[e] == sp.mobius(e) for e in D)
mu6 = mobius(D, lambda a, b: b % a == 0, 6)
assert all(mu6[e] == sp.mobius(e // 6) for e in mu6)
print('Boolean lattice and divisors of 360: Proposition 3.4 confirmed')

def hall(elements, leq, x, y):
    # alternating count of chains x = x0 < x1 < ... < xk = y
    inner = [z for z in elements if leq(x, z) and leq(z, y)]
    @lru_cache(None)
    def chains(a):   # signed number of chains from a to y
        if a == y: return 1
        return -sum(chains(b) for b in inner if b != a and leq(a, b))
    return chains(x)
assert all(hall(D, lambda a, b: b % a == 0, 1, e) == sp.mobius(e) for e in D)
print("Philip Hall's formula agrees on the divisors of 360")

def set_partitions(s):
    s = list(s)
    if not s: yield frozenset(); return
    first, rest = s[0], s[1:]
    for p in set_partitions(rest):
        blocks = list(p)
        yield frozenset(blocks + [frozenset([first])])
        for i in range(len(blocks)):
            yield frozenset(blocks[:i] + [blocks[i] | {first}] + blocks[i + 1:])

def refines(p, q):
    return all(any(b <= c for c in q) for b in p)

for n in range(1, 7):
    P = list(set_partitions(range(n)))
    bottom = frozenset(frozenset([i]) for i in range(n)); top = frozenset([frozenset(range(n))])
    t0 = time.time(); mu = mobius(P, refines, bottom)
    print(f"Pi_{n}: {len(P):3} partitions (Bell number), mu(0,1) = {mu[top]:5}, (-1)^(n-1)(n-1)! = {(-1) ** (n - 1) * math.factorial(n - 1):5}")

The sizes $1,2,5,15,52,203$ are the Bell numbers (Chapter 4 finds their generating function), and $\mu(\hat0,\hat1)=(-1)^{n-1}(n-1)!$ as guessed in Exercise 3.2.

### P3.3
**How to find it.** A word is primitive if no rotation by a proper divisor of its length fixes it (Lemma 3.6). Duval's algorithm (1983) lists Lyndon words in lexicographic order: increment the last letter that can be incremented, then extend the word periodically to length $n$; it outputs a Lyndon word whenever the current length divides $n$. For polynomials over $\mathbb F_2$, store them as integers (bit $i$ is the coefficient of $x^i$) and test irreducibility by trial division.

In [ ]:
def is_primitive(w):
    n = len(w)
    return all(w[d:] + w[:d] != w for d in range(1, n) if n % d == 0)

def M(n, q=2):
    return int(sum(sp.mobius(n // d) * q ** d for d in sp.divisors(n)))

def lyndon_words(n, q=2):
    # Duval's algorithm: all Lyndon words of length dividing n; keep those of length n
    w = [-1]
    while w:
        w[-1] += 1
        if len(w) == n: yield ''.join(map(str, w))
        m = len(w)
        while len(w) < n: w.append(w[len(w) - m])
        while w and w[-1] == q - 1: w.pop()

def pmod(a, b):
    while a.bit_length() >= b.bit_length():
        a ^= b << (a.bit_length() - b.bit_length())
    return a

def irreducible_count(n):
    cnt = 0
    for f in range(1 << n, 1 << (n + 1)):
        if all(pmod(f, g) for g in range(2, 1 << (n // 2 + 1)) if g.bit_length() - 1 <= n // 2):
            cnt += 1
    return cnt

print(' n  primitive  M(n)  Lyndon  M(n)/n  irreducible')
for n in range(1, 13):
    prim = sum(is_primitive(''.join(w)) for w in itertools.product('01', repeat=n)) if n <= 12 else None
    L = list(lyndon_words(n))
    assert all(all(w < w[r:] + w[:r] for r in range(1, n)) for w in L)
    print(f"{n:2} {prim:9} {M(n):5} {len(L):7} {M(n) // n:7} {irreducible_count(n):12}")
print('Lyndon words of length 6:', list(lyndon_words(6)))

### P3.4
**How to find it.** Encoding: keep the current leaves in a min-heap; pluck the smallest, record its neighbour, and push the neighbour if its degree drops to $1$. Decoding: by Lemma 3.8 the degrees are known in advance ($1+$ number of occurrences), so the same heap of current leaves replays the plucking. Both run in $O(n\log n)$.

In [ ]:
def prufer_encode(n, edges):
    adj = [set() for _ in range(n + 1)]
    for a, b in edges: adj[a].add(b); adj[b].add(a)
    deg = [len(s) for s in adj]
    heap = [v for v in range(1, n + 1) if deg[v] == 1]; heapq.heapify(heap)
    code = []
    for _ in range(n - 2):
        leaf = heapq.heappop(heap)
        nb = next(iter(adj[leaf]))
        code.append(nb); adj[nb].discard(leaf); deg[nb] -= 1
        if deg[nb] == 1: heapq.heappush(heap, nb)
    return code

def prufer_decode(code):
    n = len(code) + 2
    deg = [1] * (n + 1); deg[0] = 0
    for a in code: deg[a] += 1
    heap = [v for v in range(1, n + 1) if deg[v] == 1]; heapq.heapify(heap)
    edges = []
    for a in code:
        leaf = heapq.heappop(heap)
        edges.append((leaf, a)); deg[a] -= 1
        if deg[a] == 1: heapq.heappush(heap, a)
    edges.append(tuple(heapq.heappop(heap) for _ in range(2)))
    return edges

print('the tree of Figure 3.2 has code', prufer_encode(7, [(1, 4), (2, 4), (3, 5), (4, 5), (5, 6), (6, 7)]))
for n in range(2, 8):
    trees = set()
    for code in itertools.product(range(1, n + 1), repeat=n - 2):
        E = prufer_decode(list(code))
        assert prufer_encode(n, E) == list(code)
        trees.add(frozenset(frozenset(e) for e in E))
    print(f"n={n}: {len(trees):6} distinct trees, n^(n-2) = {n ** (n - 2)}")

In [ ]:
n = 10**5
deg_counts = Counter()
leaf_fracs = []
for trial in range(5):
    code = [random.randint(1, n) for _ in range(n - 2)]
    deg = Counter(code)
    leaf_fracs.append(sum(1 for v in range(1, n + 1) if deg[v] == 0) / n)
    deg_counts.update(deg[v] + 1 for v in range(1, n + 1))
print('fraction of leaves:', [round(f, 4) for f in leaf_fracs], ' 1/e =', round(1 / math.e, 4))
tot = sum(deg_counts.values())
print('degree  frequency  Poisson(1) at degree-1')
for d in range(1, 8):
    print(f"{d:6} {deg_counts[d] / tot:10.4f} {math.exp(-1) / math.factorial(d - 1):10.4f}")

By Lemma 3.8 the degree of a vertex is $1$ plus the number of times its letter appears among $n-2$ uniform letters, a Binomial$(n-2,\frac1n)$ variable, which tends to Poisson$(1)$ (Tome 3). A random tree is a Poisson$(1)$ branching process in disguise, conditioned to have $n$ individuals: this is the starting point of the theory of random trees (Aldous's continuum random tree).

### P3.5
**How to find it.** A sequence of $m$ numbers of children with sum $m-1$ is a composition, which stars and bars samples uniformly. By the cycle lemma exactly one of its $m$ rotations is a Łukasiewicz word (Proposition 3.13), and the rotations are distinct (a periodic sequence of period $d<m$ would have sum divisible by $m/d>1$). So the map "rotate to the valid shift" is exactly $m$-to-one onto plane trees, and a uniform composition gives a uniform tree.

In [ ]:
def good_shifts(x):
    m = len(x); S = list(itertools.accumulate(x + x))
    return [i for i in range(m) if all(S[j] - (S[i - 1] if i else 0) > 0 for j in range(i, i + m))]

for trial in range(2000):
    m = random.randint(1, 12)
    x = [random.randint(-4, 1) for _ in range(m)]
    k = sum(x)
    if k >= 1: assert len(good_shifts(x)) == k
print('cycle lemma confirmed on random sequences')
print('Figure 3.3 sequence: good shifts start after positions', good_shifts([1, -2, 1, 1, -1, 1, 1]))

def random_plane_tree(m):
    w = [1] * (m - 1) + [0] * (m - 1); random.shuffle(w)   # stars and bars: a uniform composition
    c, run = [], 0
    for b in w:
        if b: run += 1
        else: c.append(run); run = 0
    c.append(run)                                          # m parts with sum m - 1
    x = [1 - ci for ci in reversed(c)]                     # reversed steps have positive partial sums when valid
    i = good_shifts(x)[0]                                  # the unique good shift (k = 1)
    y = x[i:] + x[:i]
    return tuple(1 - v for v in reversed(y))

def is_lukasiewicz(c):
    h = 1
    for j, ci in enumerate(c):
        h += ci - 1
        if h < 1 and j < len(c) - 1: return False
    return h == 0

cnt = Counter(random_plane_tree(5) for _ in range(70000))
assert all(is_lukasiewicz(c) for c in cnt)
print(f'{len(cnt)} plane trees with 5 vertices (C_4 = 14); frequencies between {min(cnt.values())} and {max(cnt.values())} (expected 5000 each)')

ARITY = {'+': 2, '*': 2, 'neg': 1}
def parse_prefix(tokens):
    h = 1
    for j, t in enumerate(tokens):
        if h < 1: return f'rejected: extra token {t!r} at position {j}'
        h += ARITY.get(t, 0) - 1
    return 'accepted' if h == 0 else f'rejected: {h} operand(s) missing'
for e in ['+ * a b c', '+ a', '* + a b neg c', 'a b']:
    print(f'{e:16} {parse_prefix(e.split())}')

All $14$ trees appear, with frequencies within the expected random fluctuations of about $\sqrt{5000}\approx70$ around $5000$. The counter $h_j$ is the parser's stack depth: the number of operands still expected.

### P3.6
**How to find it.** For a permutation $\sigma$ of $\{0,\dots,n-1\}$, follow the pairs $\{a,b\}\mapsto\{\sigma a,\sigma b\}$ and record the cycle lengths. Summing $\prod_j(1+x^j)^{c_j}$ over all $\sigma$ and dividing by $n!$ is Pólya's theorem with weight $x$ per edge. For the brute force, the canonical form of a graph is the smallest relabelled edge set.

In [ ]:
x = sp.symbols('x')
def pair_cycle_type(sigma):
    n = len(sigma); seen = set(); ct = Counter()
    for a, b in itertools.combinations(range(n), 2):
        if (a, b) in seen: continue
        L, p = 0, (a, b)
        while p not in seen:
            seen.add(p); L += 1
            p = tuple(sorted((sigma[p[0]], sigma[p[1]])))
        ct[L] += 1
    return ct

for n in range(1, 7):
    types = Counter(tuple(sorted(pair_cycle_type(s).items())) for s in itertools.permutations(range(n)))
    poly = sp.expand(sum(c * sp.prod([(1 + x ** j) ** e for j, e in t]) for t, c in types.items()) / math.factorial(n))
    coeffs = [int(c) for c in sp.Poly(poly, x).all_coeffs()[::-1]] if n > 1 else [1]
    print(f"n={n}: {sum(coeffs):5} graphs; by number of edges {coeffs}")

def canon(n, E):
    return min(tuple(sorted(tuple(sorted((p[a], p[b]))) for a, b in E)) for p in itertools.permutations(range(n)))
for n in range(1, 6):
    pairs = list(itertools.combinations(range(n), 2))
    classes = {canon(n, [pairs[i] for i in range(len(pairs)) if mask >> i & 1]) for mask in range(1 << len(pairs))}
    print(f"brute force n={n}: {len(classes)} classes")
print('cycle types of S_5 on pairs:')
for t, c in sorted(Counter(tuple(sorted(pair_cycle_type(s).items())) for s in itertools.permutations(range(5))).items(), key=lambda z: -z[1]):
    print('  ', c, ' x ', ' '.join(f't{j}^{e}' for j, e in t))

### P3.7
**How to find it.** (a) A rooted tree has a canonical string: the sorted list of the canonical strings of its subtrees, in brackets (Aho, Hopcroft and Ullman's algorithm). Rooted trees with $n$ vertices are a root plus a multiset of rooted trees of total size $n-1$, which we can generate from the smaller ones. (b) Implement the formula of Theorem 3.16 with truncated power series; for the brute force, grow trees by adding a leaf to every vertex of degree $<4$, and canonicalise by rooting at the centroid (or at the halving edge).

In [ ]:
def t_numbers(N):
    t = [0, 1]
    for n in range(1, N):
        s = lambda m: sum(d * t[d] for d in sp.divisors(m))
        t.append(sum(s(m) * t[n + 1 - m] for m in range(1, n + 1)) // n)
    return t

def rooted_trees(N):
    # canonical strings of rooted trees by size
    T = {1: ['()']}
    for n in range(2, N + 1):
        out = set()
        def build(rem, maxkey, parts):
            if rem == 0:
                out.add('(' + ''.join(sorted(parts)) + ')'); return
            for size in range(1, rem + 1):
                for s in T[size]:
                    if (size, s) <= maxkey:
                        build(rem - size, (size, s), parts + [s])
        build(n - 1, (n, '~'), [])
        T[n] = sorted(out)
    return T

t = t_numbers(16)
RT = rooted_trees(10)
print('recursion :', t[1:])
print('generated :', [len(RT[n]) for n in range(1, 11)])

In [ ]:
def series_mul(a, b, N):
    c = [0] * (N + 1)
    for i, ai in enumerate(a[:N + 1]):
        if ai:
            for j, bj in enumerate(b[:N + 1 - i]):
                c[i + j] += ai * bj
    return c

def subs_pow(a, k, N):   # a(x^k), truncated
    c = [0] * (N + 1)
    for i, ai in enumerate(a):
        if i * k <= N: c[i * k] += ai
    return c

def ternary(N):
    r = [1] + [0] * N
    for n in range(1, N + 1):   # r_n only depends on r_0..r_{n-1}
        A = r[:n] + [0] * (N + 1 - n)
        A3 = series_mul(series_mul(A, A, N), A, N)
        A12 = series_mul(A, subs_pow(A, 2, N), N)
        A_3 = subs_pow(A, 3, N)
        r[n] = (A3[n - 1] + 3 * A12[n - 1] + 2 * A_3[n - 1]) // 6
    return r

def alkanes(n, r):
    N = n - 1
    rho = [r[m] if 2 * m < n else 0 for m in range(N + 1)]
    p1, p2, p3, p4 = (subs_pow(rho, k, N) for k in (1, 2, 3, 4))
    m = lambda *fs: (lambda acc: acc)(__import__('functools').reduce(lambda u, v: series_mul(u, v, N), fs))
    z = [a + 6 * b + 3 * c + 8 * d + 6 * e for a, b, c, d, e in zip(m(p1, p1, p1, p1), m(p1, p1, p2), m(p2, p2), m(p1, p3), p4)]
    eps = math.comb(r[n // 2] + 1, 2) if n % 2 == 0 else 0
    return z[N] // 24 + eps

r = ternary(40)
print('alcohols r_n:', r[1:16])
a = [None] + [alkanes(n, r) for n in range(1, 41)]
print('alkanes a_n :', a[1:16])
print('C20H42:', a[20], '   C30H62:', a[30], '   C40H82:', a[40])

In [ ]:
def free_canon(adj):
    # canonical string of a free tree: root at the centroid, or at the halving edge
    n = len(adj)
    def sizes_from(root):
        parent = {root: None}; order = [root]
        for v in order:
            for u in adj[v]:
                if u not in parent: parent[u] = v; order.append(u)
        size = {}
        for v in reversed(order):
            size[v] = 1 + sum(size[u] for u in adj[v] if parent.get(u) == v)
        return parent, size
    parent, size = sizes_from(0)
    def branches(v):
        return [size[u] for u in adj[v] if parent.get(u) == v] + ([n - size[v]] if parent[v] is not None else [])
    cents = [v for v in range(n) if max(branches(v), default=0) <= n / 2]
    def enc(v, p):
        return '(' + ''.join(sorted(enc(u, v) for u in adj[v] if u != p)) + ')'
    if len(cents) == 1:
        return enc(cents[0], None)
    u, v = cents
    return '|'.join(sorted([enc(u, v), enc(v, u)]))

def grow(maxdeg, N):
    level = {free_canon([[]]): [[]]}
    counts = [1]
    for n in range(2, N + 1):
        new = {}
        for adj in level.values():
            for v in range(len(adj)):
                if len(adj[v]) < maxdeg:
                    b = [list(s) for s in adj] + [[v]]
                    b[v].append(len(adj))
                    new.setdefault(free_canon(b), b)
        level = new; counts.append(len(level))
    return counts

t0 = time.time()
brute = grow(4, 12)
print('brute force alkanes:', brute, f'({time.time() - t0:.1f} s)')
print('Theorem 3.16      :', a[1:13])
assert brute == a[1:13]

Thirteen carbons was the limit of Cayley's patience; Theorem 3.16 reaches $\mathrm C_{40}$ instantly, with $62\,481\,801\,147\,341$ isomers. The brute force agrees with the formula wherever it can run, and it uses Jordan's lemma too, to make a canonical form: the two halves of a bicentroidal tree are sorted, which is the unordered pair of the proof.

### P3.8
**How to find it.** Use the $q$-Pascal recursion (Proposition 3.17) with sympy polynomials. For the brute force, a subspace of $\mathbb F_2^n$ is a set of vectors (integers below $2^n$) closed under XOR; generate it from spanning sets. For the random matrices, store each row as an integer and eliminate with XOR.

In [ ]:
q, X = sp.symbols('q X')
@lru_cache(None)
def gauss(n, k):
    if k < 0 or k > n: return sp.Integer(0)
    if k == 0 or k == n: return sp.Integer(1)
    return sp.expand(gauss(n - 1, k - 1) + q ** k * gauss(n - 1, k))

print('[6 choose 3]_q =', gauss(6, 3))
for n in range(7):
    for k in range(n + 1):
        inv = sum(q ** sum(1 for i in range(n) for j in range(i + 1, n) if w[i] == 1 and w[j] == 0)
                  for w in itertools.product([0, 1], repeat=n) if sum(w) == k)
        assert sp.expand(inv - gauss(n, k)) == 0
    lhs = sp.expand(sp.prod([1 + q ** i * X for i in range(n)]))
    rhs = sp.expand(sum(q ** (k * (k - 1) // 2) * gauss(n, k) * X ** k for k in range(n + 1)))
    assert sp.expand(lhs - rhs) == 0
print('inversion formula and q-binomial theorem checked for n <= 6')

def subspaces(n):
    found = Counter(); seen = set()
    frontier = [frozenset([0])]
    seen.add(frontier[0])
    while frontier:
        V = frontier.pop()
        found[len(V).bit_length() - 1] += 1
        for v in range(1, 1 << n):
            if v not in V:
                W = frozenset(V | {u ^ v for u in V})
                if W not in seen: seen.add(W); frontier.append(W)
    return found
for n in range(1, 6):
    f = subspaces(n)
    print(f"F_2^{n}: subspaces by dimension {[f[k] for k in range(n + 1)]}, formula {[gauss(n, k).subs(q, 2) for k in range(n + 1)]}")

In [ ]:
def rank_f2(rows):
    rows = list(rows); r = 0
    for bit in reversed(range(max(rows).bit_length() if any(rows) else 0)):
        piv = next((i for i in range(r, len(rows)) if rows[i] >> bit & 1), None)
        if piv is None: continue
        rows[r], rows[piv] = rows[piv], rows[r]
        for i in range(len(rows)):
            if i != r and rows[i] >> bit & 1: rows[i] ^= rows[r]
        r += 1
    return r

print(' n  simulated  prod(1-2^-i)')
for n in [1, 2, 3, 5, 10, 20, 60]:
    trials = 20000
    inv = sum(rank_f2([random.getrandbits(n) for _ in range(n)]) == n for _ in range(trials))
    print(f"{n:2} {inv / trials:10.4f} {math.prod(1 - 2.0 ** -i for i in range(1, n + 1)):12.6f}")
print('limit:', math.prod(1 - 2.0 ** -i for i in range(1, 200)))

### P3.9
**How to find it.** Seat the women in the even seats; a seating of the men is a permutation $\sigma$ with $\sigma(i)\notin\{i-1,i\}$ mod $n$.

In [ ]:
def menage_brute(n):
    return sum(all(s[i] != i and s[i] != (i - 1) % n for i in range(n)) for s in itertools.permutations(range(n)))
def touchard(n):
    return sum((-1) ** k * sp.Rational(2 * n, 2 * n - k) * math.comb(2 * n - k, k) * math.factorial(n - k) for k in range(n + 1))
for n in range(3, 9):
    print(n, menage_brute(n), touchard(n))

The sequence $1,2,13,80,579,4738,\dots$ grows like $n!/e^2$: each man avoids two gaps, and for large $n$ the $2n$ forbidden events behave almost independently, each of probability $\frac1n$.